In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import json

def find_notebooks_dir() -> Path:
    cwd = Path.cwd().resolve()

    for p in [cwd, *cwd.parents]:
        if (p / "dataset").exists() and (p / "figures").exists():
            return p

    for p in [cwd, *cwd.parents]:
        candidate = p / "analysis" / "notebooks"
        if candidate.exists():
            return candidate

    return cwd

NOTEBOOKS_DIR = find_notebooks_dir()
DATASET_PATH = NOTEBOOKS_DIR / "dataset" / "yandex_music_data.json"
EXPORTS_DIR = NOTEBOOKS_DIR / "exports"
FIG_DIR = NOTEBOOKS_DIR / "figures"

FIG_DIR.mkdir(parents=True, exist_ok=True)

WAVE_CSV  = EXPORTS_DIR / "reco_wave_demo.csv"
ALICE_CSV = EXPORTS_DIR / "reco_alice_demo.csv"

print("NOTEBOOKS_DIR:", NOTEBOOKS_DIR)
print("EXPORTS_DIR:", EXPORTS_DIR)
print("WAVE_CSV:", WAVE_CSV, "exists:", WAVE_CSV.exists())
print("ALICE_CSV:", ALICE_CSV, "exists:", ALICE_CSV.exists())

def savefig(name: str):
    out = FIG_DIR / name
    plt.tight_layout()
    plt.savefig(out, dpi=180, bbox_inches="tight")
    plt.close()
    return out


NOTEBOOKS_DIR: C:\Users\kesha\OneDrive\Рабочий стол\BigData\yandex-music-preference-analysis\analysis\notebooks
EXPORTS_DIR: C:\Users\kesha\OneDrive\Рабочий стол\BigData\yandex-music-preference-analysis\analysis\notebooks\exports
WAVE_CSV: C:\Users\kesha\OneDrive\Рабочий стол\BigData\yandex-music-preference-analysis\analysis\notebooks\exports\reco_wave_demo.csv exists: True
ALICE_CSV: C:\Users\kesha\OneDrive\Рабочий стол\BigData\yandex-music-preference-analysis\analysis\notebooks\exports\reco_alice_demo.csv exists: True


In [2]:
if not WAVE_CSV.exists() or not ALICE_CSV.exists():
    raise FileNotFoundError(
        "Не найдены CSV из ноутбуков 03/04.\n"
        f"Ожидаю:\n- {WAVE_CSV}\n- {ALICE_CSV}\n"
        "Сначала запусти 03 и 04 (ячейка сохранения CSV)."
    )

reco_wave = pd.read_csv(WAVE_CSV)
reco_alice = pd.read_csv(ALICE_CSV)

# гарантируем типы
for d in [reco_wave, reco_alice]:
    d["track_id"] = d["track_id"].astype("int64")
    d["is_from_likes"] = d["is_from_likes"].astype(bool)
    d["score"] = d["score"].astype(float)
    d["genre"] = d["genre"].astype(str)

# liked_ids для overlap (берём из твоего реального датасета)
if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATASET_PATH}")

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

likes = pd.DataFrame(data.get("likes", []))
likes["track_id"] = likes["track_id"].astype("int64")
liked_ids = set(likes["track_id"].tolist())

print("Wave reco:", len(reco_wave), "Alice reco:", len(reco_alice), "Liked tracks:", len(liked_ids))

reco_wave.head()


Wave reco: 80 Alice reco: 80 Liked tracks: 1026


,algo,track_id,is_from_likes,genre,score
0,Моя волна,57997910,True,rap,0.999331
1,Моя волна,656485,True,country,0.985608
2,Моя волна,17190997,True,allrock,0.978284
3,Моя волна,21719115,True,pop,0.971171
4,Моя волна,12830191,True,alternative,0.965276


In [3]:
def shannon_entropy(counts) -> float:
    p = np.array(counts, dtype=float)
    p = p[p > 0]
    if p.sum() == 0:
        return 0.0
    p = p / p.sum()
    return float(-(p * np.log2(p)).sum())

def metrics_for(reco_df: pd.DataFrame, algo_name: str) -> dict:
    reco_set = set(reco_df["track_id"].tolist())
    overlap = len(reco_set & liked_ids)
    novelty = 1 - overlap / len(reco_set) if len(reco_set) else 0.0

    genre_counts = reco_df["genre"].value_counts()
    entropy = shannon_entropy(genre_counts.values)

    return {
        "algo": algo_name,
        "n_reco": int(len(reco_set)),
        "overlap_with_likes": int(overlap),
        "repeat_rate": float(1 - novelty),
        "novelty_rate": float(novelty),
        "n_genres": int(genre_counts.shape[0]),
        "genre_entropy": float(entropy),
        "avg_score": float(reco_df["score"].mean()),
        "median_score": float(reco_df["score"].median()),
    }

m_wave = metrics_for(reco_wave, "Моя волна")
m_alice = metrics_for(reco_alice, "Алиса")

metrics_df = pd.DataFrame([m_wave, m_alice])
metrics_df


,algo,n_reco,overlap_with_likes,repeat_rate,novelty_rate,n_genres,genre_entropy,avg_score,median_score
0,Моя волна,80,40,0.5,0.5,28,4.030361,0.736065,0.831161
1,Алиса,80,16,0.2,0.8,35,4.716648,0.542307,0.581255


In [4]:
wave_ids = set(reco_wave["track_id"])
alice_ids = set(reco_alice["track_id"])

intersection = len(wave_ids & alice_ids)
only_wave = len(wave_ids - alice_ids)
only_alice = len(alice_ids - wave_ids)

plt.figure(figsize=(6,4))
ax = plt.gca()
ax.set_aspect('equal')
ax.add_patch(Circle((0.45, 0.5), 0.35, fill=False))
ax.add_patch(Circle((0.65, 0.5), 0.35, fill=False))

ax.text(0.35, 0.5, str(only_wave), ha="center", va="center", fontsize=14)
ax.text(0.55, 0.5, str(intersection), ha="center", va="center", fontsize=14)
ax.text(0.75, 0.5, str(only_alice), ha="center", va="center", fontsize=14)

ax.text(0.35, 0.82, "Моя волна", ha="center")
ax.text(0.75, 0.82, "Алиса", ha="center")

ax.set_xlim(0,1)
ax.set_ylim(0,1)
ax.axis("off")
plt.title("Пересечение рекомендаций (Wave vs Alice)")
savefig("compare_venn_wave_vs_alice.png")


WindowsPath('C:/Users/kesha/OneDrive/Рабочий стол/BigData/yandex-music-preference-analysis/analysis/notebooks/figures/compare_venn_wave_vs_alice.png')

In [5]:
g_wave = reco_wave["genre"].value_counts().head(12)
g_alice = reco_alice["genre"].value_counts().head(12)

genres_union = sorted(set(g_wave.index) | set(g_alice.index))
wave_vals = [int(g_wave.get(g, 0)) for g in genres_union]
alice_vals = [int(g_alice.get(g, 0)) for g in genres_union]

x = np.arange(len(genres_union))
w = 0.4

plt.figure(figsize=(12,5))
plt.bar(x - w/2, wave_vals, width=w, label="Моя волна")
plt.bar(x + w/2, alice_vals, width=w, label="Алиса")

plt.xticks(x, genres_union, rotation=45, ha="right")
plt.title("Сравнение жанров в рекомендациях (топ жанров)")
plt.xlabel("Жанр")
plt.ylabel("Количество треков")
plt.legend()

savefig("compare_genres_wave_vs_alice.png")


WindowsPath('C:/Users/kesha/OneDrive/Рабочий стол/BigData/yandex-music-preference-analysis/analysis/notebooks/figures/compare_genres_wave_vs_alice.png')

In [6]:
plt.figure(figsize=(8,4))
plt.bar(metrics_df["algo"], metrics_df["novelty_rate"])
plt.title("Доля новых треков (novelty rate) — сравнение алгоритмов")
plt.ylabel("Novelty rate (0..1)")
savefig("compare_novelty_wave_vs_alice.png")


WindowsPath('C:/Users/kesha/OneDrive/Рабочий стол/BigData/yandex-music-preference-analysis/analysis/notebooks/figures/compare_novelty_wave_vs_alice.png')

In [7]:
plt.figure(figsize=(8,4))
plt.boxplot([reco_wave["score"], reco_alice["score"]], labels=["Моя волна", "Алиса"])
plt.title("Сравнение распределения score")
plt.ylabel("Score")
savefig("compare_score_boxplot.png")


WindowsPath('C:/Users/kesha/OneDrive/Рабочий стол/BigData/yandex-music-preference-analysis/analysis/notebooks/figures/compare_score_boxplot.png')

In [8]:
def pct(x):
    return f"{x*100:.1f}%"

print("СРАВНЕНИЕ (на основе данных из 03/04):")
print(
    f"- Моя волна: novelty={pct(m_wave['novelty_rate'])}, повторы={pct(m_wave['repeat_rate'])}, "
    f"жанров={m_wave['n_genres']}, энтропия={m_wave['genre_entropy']:.2f}, overlap_with_likes={m_wave['overlap_with_likes']}."
)
print(
    f"- Алиса: novelty={pct(m_alice['novelty_rate'])}, повторы={pct(m_alice['repeat_rate'])}, "
    f"жанров={m_alice['n_genres']}, энтропия={m_alice['genre_entropy']:.2f}, overlap_with_likes={m_alice['overlap_with_likes']}."
)

if m_alice["novelty_rate"] > m_wave["novelty_rate"]:
    print("\nВывод: по метрике novelty Алиса предлагает больше новых треков, чем Моя волна.")
else:
    print("\nВывод: по метрике novelty Моя волна предлагает больше новых треков, чем Алиса.")

if m_alice["genre_entropy"] > m_wave["genre_entropy"]:
    print("Также у Алисы выше жанровое разнообразие (энтропия жанров больше).")
else:
    print("Также у Моей волны выше жанровое разнообразие (энтропия жанров больше).")


СРАВНЕНИЕ (на основе данных из 03/04):
- Моя волна: novelty=50.0%, повторы=50.0%, жанров=28, энтропия=4.03, overlap_with_likes=40.
- Алиса: novelty=80.0%, повторы=20.0%, жанров=35, энтропия=4.72, overlap_with_likes=16.

Вывод: по метрике novelty Алиса предлагает больше новых треков, чем Моя волна.
Также у Алисы выше жанровое разнообразие (энтропия жанров больше).
